In [21]:
import numpy as np
import pandas as pd

from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

In [22]:
df = pd.read_csv("bp_data.csv")
df.head()

,bp,age,score,sex
0,124,22,65,m
1,117,33,34,w
2,127,35,35,m
3,132,44,43,m
4,117,42,54,w


In [23]:
df["sex"].unique()

array(['m', 'w'], dtype=object)

In [24]:
df["sex"] = df["sex"].map({"m":1,"w":0})
df.head()

,bp,age,score,sex
0,124,22,65,1
1,117,33,34,0
2,127,35,35,1
3,132,44,43,1
4,117,42,54,0


In [32]:
X = df.drop("bp",axis=1)
y = df["bp"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X.shape

(8, 3)

In [38]:
p = X_train.shape[1]

k_models = []

for k in range(1,p+1):
    best_rss = float("inf")
    best_features = None
    
    for features in combinations(X_train.columns,k):
        model = LinearRegression()
        model.fit(X_train[list(features)],y_train)
        y_pred = model.predict(X_test[list(features)])
        
        curr_rss = np.sum((y_test-y_pred)**2)
        
        if curr_rss < best_rss:
            best_rss = curr_rss
            best_features = features

    k_models.append((k,best_rss,p,best_features))

models = {"k_value":[],"total_features":[],"features":[],"Rss":[]}
for k,rss,p,features in k_models:
    models["k_value"].append(k)
    models["features"].append(features)
    models["Rss"].append(rss)
    models["total_features"].append(p)

models = pd.DataFrame(models)
models

,k_value,total_features,features,Rss
0,1,3,"(sex,)",17.000000
1,2,3,"(age, sex)",25.675208
2,3,3,"(age, score, sex)",19.864613


In [43]:
best_model = None
best_adj_Rsquare = None
n = X_test.shape[0]

for k in models["k_value"].values:
    adjusted_Rsquare = float("-inf")
    for features in models["features"].values:
        model = LinearRegression()
        model.fit(X_train[list(features)],y_train)
        y_pred = model.predict(X_test[list(features)])
        
        rss = np.sum((y_pred-y_test)**2)
        tss = np.sum((y_test-y_test.mean())**2)
        
        rSquare = 1 - (rss/tss)
        
        adj_rSquare = 1 - ((1-rSquare)*(n-1)/(n-len(features)-1))
        
        if adj_rSquare > adjusted_Rsquare:
            best_adj_Rsquare = adj_rSquare
            adjusted_Rsquare = adj_rSquare
            best_features = features

print(best_features)

('age', 'sex')


C:\Users\SILICON\AppData\Local\Temp\ipykernel_4484\3386721898.py:17: RuntimeWarning: divide by zero encountered in double_scalars
  adj_rSquare = 1 - ((1-rSquare)*(n-1)/(n-len(features)-1))
C:\Users\SILICON\AppData\Local\Temp\ipykernel_4484\3386721898.py:17: RuntimeWarning: divide by zero encountered in double_scalars
  adj_rSquare = 1 - ((1-rSquare)*(n-1)/(n-len(features)-1))
C:\Users\SILICON\AppData\Local\Temp\ipykernel_4484\3386721898.py:17: RuntimeWarning: divide by zero encountered in double_scalars
  adj_rSquare = 1 - ((1-rSquare)*(n-1)/(n-len(features)-1))
